In [ ]:
%pip install -q -U google-genai python-dotenv pandas tqdm rapidfuzz


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key: 
    raise ValueError("GEMINI_API_KEY not found!")

client = genai.Client(api_key=api_key)

print("Gemini ready ✅")

Gemini ready ✅


In [19]:
DICTIONARY_PATH = "Dictionary.txt"
WORD_LIST_PATH = "final_word_based_corpus.txt"
COMMON_FREQ_PATH = "wordfreq_si_clean.txt"
CORPUS_PATH = "sinhala_large_corpus.txt"

This section identifies and extracts only Sinhala words from the input text for further processing and validation.ignores/removes non-Sinhala words and extracts only Sinhala words from the text

In [20]:
import re
from collections import Counter
from rapidfuzz import process, fuzz

SINHALA_WORD_RE = re.compile(r"[\u0D80-\u0DFF]+")

def extract_sinhala_words(text):
    return SINHALA_WORD_RE.findall(text)

This section loads the Sinhala dictionary, corpus, and frequency datasets, then combines them to create a valid Sinhala vocabulary and frequency database used for word validation and correction ranking.

In [21]:
def load_word_list(path):
    words = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            words.update(extract_sinhala_words(line))
    return words

def load_frequency_words(path):
    freq = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2 and parts[1].isdigit():
                freq[parts[0]] = int(parts[1])
    return freq

def load_corpus_words(path):
    counter = Counter()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            counter.update(extract_sinhala_words(line))
    return dict(counter)

# Load everything
dictionary_words = load_word_list(DICTIONARY_PATH)
final_word_words = load_word_list(WORD_LIST_PATH)
common_frequency_words = load_frequency_words(COMMON_FREQ_PATH)
corpus_frequency_words = load_corpus_words(CORPUS_PATH)

# Combine frequencies
combined_frequency = Counter()
combined_frequency.update(common_frequency_words)
combined_frequency.update(corpus_frequency_words)

# Valid words
valid_words = (
    dictionary_words
    | final_word_words
    | set(common_frequency_words.keys())
    | set(corpus_frequency_words.keys())
)

print("Dictionary:", len(dictionary_words))
print("Final word list:", len(final_word_words))
print("Common freq:", len(common_frequency_words))   # from :contentReference[oaicite:0]{index=0}
print("Corpus:", len(corpus_frequency_words))        # from :contentReference[oaicite:1]{index=1}
print("Total valid:", len(valid_words))

Dictionary: 50071
Final word list: 3208
Common freq: 50000
Corpus: 166
Total valid: 51755


This section cleans the Gemini output and uses the loaded Sinhala datasets to validate words, detect unknown words, and identify low-frequency suspicious words

In [35]:
def has_english_letters(text):
    return bool(re.search(r"[A-Za-z]", text))

def clean_extra_spaces(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?।])", r"\1", text)
    return text.strip()

def sinhala_only_clean(text):

    # Remove English letters
    text = re.sub(r"[A-Za-z]", "", text)

    # Remove long number sequences like phone numbers
    text = re.sub(r"\d[\d\s-]{5,}", "", text)

    return clean_extra_spaces(text)
def get_word_score(word):
    return combined_frequency.get(word, 0)

def get_unknown_words(text):
    words = extract_sinhala_words(text)
    return sorted(set([w for w in words if w not in valid_words]))

def get_low_frequency_words(text, min_score=2):
    words = extract_sinhala_words(text)
    return sorted(
        [(w, get_word_score(w)) for w in set(words) if get_word_score(w) < min_score],
        key=lambda x: x[1]
    )

 This section uses RapidFuzz and the Sinhala datasets to find and replace unknown or misspelled words with the most similar and highest-frequency valid Sinhala words.
 RapidFuzz is a library used to:
 find words that are similar to each other.

 if Gemini outputs a wrong or unknown Sinhala word, RapidFuzz searches the dataset and finds the closest valid Sinhala word.

In [36]:
def get_best_match(word, choices, score_cutoff=80):

    matches = process.extract(
        word,
        choices,
        scorer=fuzz.ratio,
        limit=5
    )

    best_word = word
    best_score = 0

    for match_word, similarity_score, _ in matches:

        # Skip words containing numbers
        if any(char.isdigit() for char in match_word):
            continue

        frequency_score = get_word_score(match_word)

        combined_score = similarity_score + (frequency_score * 0.0001)

        if (
            similarity_score >= score_cutoff
            and combined_score > best_score
        ):

            best_score = combined_score
            best_word = match_word

    return best_word

def correct_unknown_words(text):

    words = text.split()

    corrected_words = []

    for word in words:

        sinhala_parts = extract_sinhala_words(word)

        if not sinhala_parts:
            corrected_words.append(word)
            continue

        clean_word = sinhala_parts[0]

        if clean_word not in valid_words:

            best_match = get_best_match(clean_word, valid_words)

            corrected_words.append(
                word.replace(clean_word, best_match)
            )

        else:
            corrected_words.append(word)

    return " ".join(corrected_words)

This section defines the Gemini-based correction engine. It uses a detailed system prompt with correction rules and examples to guide Gemini in correcting noisy Sinhala text, while preserving the original meaning and conversational style. The function also handles retries, output cleaning, and validation before returning the corrected Sinhala text.

In [37]:
import time
from google.genai import types

MODEL_NAME = "models/gemini-2.5-flash"

SYSTEM_PROMPT = """
ROLE:
You are an expert Sinhala noisy-text spell and grammar correction engine specialized in correcting ASR-style and conversational Sinhala text.

TASK:
Correct noisy Sinhala text containing:

* spelling mistakes
* joined words
* broken words
* spacing mistakes
* speech-to-text transcription errors
* minor grammar mistakes

IMPORTANT RULES:

* preserve the original meaning exactly
* do NOT summarize
* do NOT remove sentences
* do NOT rewrite the text unnecessarily
* do NOT invent names, places, or new information
* correct conservatively
* if uncertain, keep the original wording
* prefer common modern Sinhala words over rare or literary words
* preserve conversational spoken style
* output Sinhala text ONLY
* no explanations
* no bullet points
* no English letters

CORRECTION PRIORITY:

1. Separate incorrectly joined words
2. Fix spacing mistakes
3. Correct obvious spelling mistakes
4. Correct obvious speech-to-text errors
5. Fix only necessary grammar issues

EXAMPLES:

Input:
කපල නෑ

Output:
කපලා නෑ

Input:
පියෝටීවිසහ

Output:
පියෝටීවී සහ

Input:
ගෙවන්නතියනේ

Output:
ගෙවන්න තියෙන්නේ

Input:
අරකළුපාඩ

Output:
අර කළු පාට

Input:
මොකද්වෙලා

Output:
මොකක්ද වෙලා

Input:
ලයින්එකේ

Output:
ලයින් එකේ

Input:
වැඩකරනා

Output:
වැඩ කරන

Input:
හරිමේ

Output:
හරි මේ

Input:
කවුලලිව

Output:
කවුරු හරි

Input:
පශික්කරනකන්

Output:
පරීක්ෂා කරනකන්
"""


def correct_with_gemini(text, max_retries=3):

    if not text.strip():
        return ""

    prompt = f"{SYSTEM_PROMPT}\n\n{text}"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=8000
                )
            )

            output = response.text.strip()
            output = sinhala_only_clean(output)

            if output and not has_english_letters(output):
                return output

        except Exception as e:
            print("Retry...", e)
            time.sleep(2 ** attempt)

    return "[ERROR]"

his section splits large noisy Sinhala text into smaller chunks for more accurate Gemini processing, then combines the corrected chunks, applies dataset-based unknown-word correction, and finally returns the corrected text along with detected unknown and low-frequency words for validation.

In [38]:
def split_text_into_chunks(text, chunk_size=250):

    words = text.split()

    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:

        current_length += len(word) + 1

        if current_length > chunk_size:

            chunks.append(" ".join(current_chunk))

            current_chunk = [word]

            current_length = len(word)

        else:
            current_chunk.append(word)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def correct_sinhala_text(text):

    chunks = split_text_into_chunks(text)

    corrected_chunks = []

    for chunk in chunks:

        corrected_chunk = correct_with_gemini(chunk)

        corrected_chunks.append(corrected_chunk)

    corrected = " ".join(corrected_chunks)

    corrected = correct_unknown_words(corrected)

    return {
        "input": text,
        "corrected_text": corrected,
        "unknown_words": get_unknown_words(corrected),
        "low_frequency_words": get_low_frequency_words(corrected)
    }

This section tests the complete Sinhala correction pipeline using a noisy Sinhala paragraph and displays the original input, corrected output, detected unknown words, and low-frequency suspicious words for evaluation.

In [39]:
test_text = """
න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ පොඩ්ඩක් බලන්න කියන  මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න
"""
result = correct_sinhala_text(test_text)

print("INPUT:\n", result["input"])
print("\nCORRECTED:\n", result["corrected_text"])
print("\nUNKNOWN:", result["unknown_words"])
print("\nLOW FREQ:", result["low_frequency_words"])



INPUT:
 
න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ පොඩ්ඩක් බලන්න කියන  මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න


CORRECTED:
 නෑ ඔන් එකේ තියෙන්නේ කපලා නෑ. පියෝටීවී සහ වොයිස් දෙක ඕන කියලා තියෙන්නේ සර්. මොකක්ද වෙලා තියෙන්නේ? වැඩ කරන පියෝටීවී එකත්, වැඩ කරන වොයිස් එකත්. හරි, මේ ලයින් එකේ ඒ පහසුකම් දෙක ඕන කරන ගැටලුවක් නෑ. ලයින් සර්, ගෙවන්න තියෙන එච්චර ගෙවන්න තියෙන්නේ සර්. පොඩ්ඩක් බලන්න කියන මේ සමහරක් ගලට මේ කේබල් බහිනකොට ගලවනවානේ අර කළු පාට බොක්ස් නවා නැති ගෑවිලා තියෙන හතරෙන් කවුරු හරි පොඩ්ඩක් හමට ගෑවිලා ඇති පරීක්ෂා කරනකන් මේ පත්තනං ඔව් කෙසේ හරිසහරිවකස එසටමත සවස් සබදතාවක් ස ඇමතිමකිසනා රඳා ඉන්න

UNKNOWN: ['ඇමතිමකිසනා', 'එසටමත', 'පත්තන